In [ ]:
%pip install duckdb pandas pyarrow numpy scipy statsmodels

In [ ]:
import os
import json
import time
import shutil
import logging
import itertools
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.stats import zipf
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
FACTORS = {
    'A_Storage_Mode': ['Parquet', 'Materialized'],  # 1=Parquet, 2=Materialized
    'B_ID_Type': ['Sequential_INTEGER', 'UUIDv4', 'UUIDv7'],  # 1=Sequential, 2=UUIDv4, 3=UUIDv7
    'C_Row_Group_Size': [32_000, 128_000, 512_000],  # 1=32K, 2=128K, 3=512K
    'D_Threads': [1, 4, 8],  # 1=1, 2=4, 3=8
    'E_Index_Strategy': ['No_Index', 'Index_On_Load', 'Index_On_Demand'],  # 1=No, 2=OnLoad, 3=OnDemand
    'F_Compression': ['Snappy', 'ZSTD', 'None'],  # 1=Snappy, 2=ZSTD, 3=None
    'G_Key_Distribution': ['Uniform', 'Zipf', 'Empirical'],  # 1=Uniform, 2=Zipf, 3=Empirical
}

L18_DESIGN_MATRIX = [
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 1],
    [0, 0, 2, 2, 2, 2, 2],
    [0, 1, 0, 1, 2, 2, 2],
    [0, 1, 1, 2, 0, 0, 0],
    [0, 1, 2, 0, 1, 1, 1],
    [0, 2, 0, 2, 1, 2, 1],
    [0, 2, 1, 0, 2, 0, 2],
    [0, 2, 2, 1, 0, 1, 0],
    [1, 0, 0, 2, 2, 1, 0],
    [1, 0, 1, 0, 0, 2, 1],
    [1, 0, 2, 1, 1, 0, 2],
    [1, 1, 0, 0, 1, 0, 2],
    [1, 1, 1, 1, 2, 1, 0],
    [1, 1, 2, 2, 0, 2, 1],
    [1, 2, 0, 1, 0, 1, 2],
    [1, 2, 1, 2, 1, 2, 0],
    [1, 2, 2, 0, 2, 0, 1]
]

NUM_ROWS = 1000000
DATA_SCHEMA = {
    "id": "int64",
    "customer_id": "int64",
    "category": "int32",
    "amount": "float64",
    "ts": "timestamp[ns]",
    "region": "int32",
    "comment": "string"
}
RANDOM_SEED = 12345
DATA_DIR = Path("./data")
RESULTS_DIR = Path("./results")

WORKLOAD_QUERIES = {
    "point_lookup": "SELECT * FROM sales_table WHERE id = {id_val};",
    "range_query": "SELECT * FROM sales_table WHERE id BETWEEN {id_val_a} AND {id_val_b};",
    "aggregation": "SELECT category, SUM(amount) FROM sales_table GROUP BY category;",
    "join": "SELECT * FROM sales_table s JOIN customers c USING (customer_id);",
    "window": "SELECT id, ROW_NUMBER() OVER (PARTITION BY category ORDER BY ts DESC) FROM sales_table;"
}

N_WARMUP_REPS = 3
N_MEASURED_REPS = 8  # 5 warm + 3 cold

METRICS_TO_COLLECT = {
    "latency_ms": {'goal': 'SIB'},
    "cpu_time_ms": {'goal': 'SIB'},
    "materialization_time_ms": {'goal': 'SIB'},
    "index_build_time_ms": {'goal': 'SIB'},
    # 'peak_memory_mb': {'goal': 'SIB'},
}

In [ ]:
def get_sn_sib(y_values):
    y = np.array(y_values)
    mean_sq = np.mean(np.square(y))
    if mean_sq == 0:
        return np.inf
    return -10 * np.log10(mean_sq)

def get_sn_lib(y_values):
    y = np.array(y_values)
    if np.any(y <= 0):
        y[y <= 0] = 1e-9

    mean_inv_sq = np.mean(1.0 / np.square(y))
    if mean_inv_sq == 0:
        return -np.inf
    return -10 * np.log10(mean_inv_sq)

def generate_dataset(run_id: int, config: dict) -> tuple[Path, dict]:
    print(f"  Generating dataset for Run {run_id}...")
    rng = np.random.default_rng(RANDOM_SEED + run_id)

    df = pd.DataFrame()

    id_type = config['B_ID_Type']

    if id_type == 'Sequential_INTEGER':
        df['id'] = np.arange(1, NUM_ROWS + 1)
    elif id_type == 'UUIDv4':
        df['id'] = [
            f"{rng.bytes(4).hex()}-{rng.bytes(2).hex()}-4{rng.bytes(1).hex()[1:]}-{rng.bytes(1).hex()}-{rng.bytes(6).hex()}"
            for _ in range(NUM_ROWS)
        ]
        df['id'] = df['id'].astype('string')
    elif id_type == 'UUIDv7':
        start_ts = int(time.time() * 1000)
        ids = []
        for i in range(NUM_ROWS):
            ts_ms = start_ts + i
            ts_high = (ts_ms >> 16) & 0xFFFFFFFF
            ts_low = ts_ms & 0xFFFF
            rand_a = rng.integers(0, 2**16)

            rand_b = int.from_bytes(rng.bytes(8), "big")

            ids.append(f"{ts_high:08x}-{ts_low:04x}-7{rand_a:03x}-{rand_b:016x}"[:36])
        df['id'] = ids
        df['id'] = df['id'].astype('string')

    dist = config['G_Key_Distribution']
    if dist == 'Uniform':
        df['customer_id'] = rng.integers(1, NUM_ROWS // 10, size=NUM_ROWS)
    elif dist == 'Zipf':
        a = 1.2
        vals = zipf.rvs(a, size=NUM_ROWS, random_state=rng)
        df['customer_id'] = vals
    elif dist == 'Empirical':
        n_hot = NUM_ROWS // 50
        n_cold = NUM_ROWS // 10
        hot_cust = rng.integers(1, n_hot, size=int(NUM_ROWS * 0.8))
        cold_cust = rng.integers(n_hot, n_cold, size=int(NUM_ROWS * 0.2))
        cust_ids = np.concatenate([hot_cust, cold_cust])
        rng.shuffle(cust_ids)
        df['customer_id'] = cust_ids

    df['category'] = rng.integers(1, 100, size=NUM_ROWS)
    df['amount'] = rng.normal(loc=100, scale=25, size=NUM_ROWS).round(2)
    df['ts'] = pd.to_datetime(rng.integers(1.6e9, 1.7e9, size=NUM_ROWS), unit='s')
    df['region'] = rng.integers(1, 10, size=NUM_ROWS)
    df['comment'] = "sample comment " + df['id'].astype(str)

    random_idx = rng.integers(0, len(df))
    point_id_val = df['id'].iloc[random_idx]
    sorted_ids = df['id'].sort_values().values

    start_idx = rng.integers(0, len(df) - 1005)
    range_start_val = sorted_ids[start_idx]
    range_end_val = sorted_ids[start_idx + 1000]
    query_params = {
        'point_id': point_id_val,
        'range_start': range_start_val,
        'range_end': range_end_val
    }

    parquet_path = DATA_DIR / f"run_{run_id}_data.parquet"
    compression = config['F_Compression']

    if compression == 'None':
        compression = None

    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(
        table,
        parquet_path,
        row_group_size=config['C_Row_Group_Size'],
        compression=compression,
        use_dictionary=False,
        write_statistics=True
    )

    return parquet_path, query_params

def get_benchmark_queries(config: dict, parquet_path: Path, query_params: dict) -> dict:
    storage_mode = config['A_Storage_Mode']
    if storage_mode == 'Parquet':
        table_name = f"read_parquet('{parquet_path.as_posix()}')"
    else:
        table_name = "sales"

    def fmt_val(val):
        if isinstance(val, (str, np.str_)):
            return f"'{val}'"
        return str(val)

    id_val = fmt_val(query_params['point_id'])
    id_val_a = fmt_val(query_params['range_start'])
    id_val_b = fmt_val(query_params['range_end'])

    queries = {}
    for name, query_template in WORKLOAD_QUERIES.items():
        q = query_template.replace("sales_table", table_name)
        q = q.replace("{id_val}", id_val)
        q = q.replace("{id_val_a}", id_val_a)
        q = q.replace("{id_val_b}", id_val_b)
        queries[name] = q

    return queries

def run_benchmark(run_id: int, config: dict, parquet_path: Path, query_params: dict) -> pd.DataFrame:
    print(f"  Running benchmark for Run {run_id}...")

    # List to store results dictionaries
    all_results = []

    db_path = DATA_DIR / f"run_{run_id}.duckdb"
    if db_path.exists():
        db_path.unlink()
    con = duckdb.connect(database=db_path.as_posix(), read_only=False)

    con.sql(f"PRAGMA threads={config['D_Threads']}")

    materialization_time_ms = 0.0
    index_build_time_ms = 0.0

    try:
        con.sql("CREATE TABLE customers (customer_id BIGINT PRIMARY KEY, name VARCHAR, tier VARCHAR);")
        con.sql("INSERT INTO customers SELECT i, 'Cust_' || i::VARCHAR, 'Gold' FROM range(1, 20000) t(i);")

        if config['A_Storage_Mode'] == 'Materialized':
            print("    Mode: Materialized. Ingesting Parquet...")
            t_start = time.perf_counter()
            con.sql(f"CREATE TABLE sales AS SELECT * FROM read_parquet('{parquet_path.as_posix()}');")
            t_end = time.perf_counter()
            materialization_time_ms = (t_end - t_start) * 1000

            if config['E_Index_Strategy'] == 'Index_On_Load':
                print("    Index: Eager (On Load). Building index now...")
                t_start = time.perf_counter()
                con.sql("CREATE INDEX idx_sales_id ON sales(id);")
                t_end = time.perf_counter()
                index_build_time_ms = (t_end - t_start) * 1000

        if config['A_Storage_Mode'] == 'Materialized' and config['E_Index_Strategy'] == 'Index_On_Demand':
            print("    Index: Lazy (On Demand). Building index now...")
            t_start = time.perf_counter()
            con.sql("CREATE INDEX idx_sales_id ON sales(id);")
            t_end = time.perf_counter()
            index_build_time_ms = (t_end - t_start) * 1000

        queries = get_benchmark_queries(config, parquet_path, query_params)

        for query_name, query_sql in queries.items():
            print(f"    Query: {query_name}")

            # Warm-up Reps
            for _ in range(N_WARMUP_REPS):
                con.sql(query_sql).fetchall()

            # Measured Reps
            for rep in range(N_MEASURED_REPS):

                t_wall_start = time.perf_counter()
                t_cpu_start = time.process_time()

                con.sql(query_sql).fetchall()

                t_cpu_end = time.process_time()
                t_wall_end = time.perf_counter()

                latency_ms = (t_wall_end - t_wall_start) * 1000
                cpu_time_ms = (t_cpu_end - t_cpu_start) * 1000

                all_results.append({
                    "run_id": run_id,
                    "query_name": query_name,
                    "rep": rep,
                    "latency_ms": latency_ms,
                    "cpu_time_ms": cpu_time_ms,
                    "materialization_time_ms": materialization_time_ms if rep == 0 else 0,
                    "index_build_time_ms": index_build_time_ms if rep == 0 else 0,
                })
                materialization_time_ms = 0
                index_build_time_ms = 0

    except duckdb.Error as e:
        print(f"!! ERROR in Run {run_id} with query {query_name}: {e}")
        all_results.append({
            "run_id": run_id,
            "query_name": query_name,
            "rep": 0,
            "latency_ms": np.nan,
            "cpu_time_ms": np.nan,
            "materialization_time_ms": np.nan,
            "index_build_time_ms": np.nan,
        })
    finally:
        con.close()
        if db_path.exists():
            db_path.unlink()

    return pd.DataFrame(all_results)

def save_results(run_id: int, config: dict, df_metrics: pd.DataFrame):
    run_dir = RESULTS_DIR / f"run_{run_id:02d}"
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / "config.json", 'w') as f:
        json.dump(config, f, indent=2)

    df_metrics.to_csv(run_dir / "metrics.csv", index=False)

def build_l18_run_plan() -> list[dict]:
    plan = []
    factor_names = list(FACTORS.keys())

    for run_idx, run_levels in enumerate(L18_DESIGN_MATRIX):
        config = {"run_id": run_idx + 1}
        for i, factor_name in enumerate(factor_names):
            level_index = run_levels[i]
            level_value = FACTORS[factor_name][level_index]
            config[factor_name] = level_value
        plan.append(config)
    return plan

def analyze_results():
    print("\n" + "="*80)
    print("Starting Statistical Analysis")
    print("="*80)

    all_metrics = []
    for run_dir in sorted(RESULTS_DIR.glob("run_*/")):
        metrics_file = run_dir / "metrics.csv"
        if metrics_file.exists():
            df = pd.read_csv(metrics_file)
            all_metrics.append(df)

    if not all_metrics:
        print("No results found. Aborting analysis.")
        return

    df_raw = pd.concat(all_metrics, ignore_index=True).dropna()
    if df_raw.empty:
        print("All results contained NaNs. Aborting analysis.")
        return

    sn_results = []
    for (run_id, query_name), group in df_raw.groupby(['run_id', 'query_name']):
        sn_row = {"run_id": run_id, "query_name": query_name}
        for metric, info in METRICS_TO_COLLECT.items():
            values = group[metric]
            if metric.endswith("_time_ms") and not metric.startswith("latency"):
                 sn_row[f"{metric}_mean"] = values[values > 0].mean()
                 continue
            if values.count() < 2:
                continue

            if info['goal'] == 'SIB':
                sn_row[f"{metric}_sn"] = get_sn_sib(values)
            elif info['goal'] == 'LIB':
                sn_row[f"{metric}_sn"] = get_sn_lib(values)
        sn_results.append(sn_row)

    df_sn = pd.DataFrame(sn_results).fillna(0)

    l18_plan = build_l18_run_plan()
    df_factors = pd.DataFrame(l18_plan)

    df_analysis = pd.merge(df_sn, df_factors, on='run_id')
    response_vars = [col for col in df_sn.columns if col.endswith('_sn')]
    factor_names = list(FACTORS.keys())

    for query_name in df_analysis['query_name'].unique():
        print(f"\n--- Analysis for Workload: {query_name} ---")
        df_query = df_analysis[df_analysis['query_name'] == query_name].copy()
        if len(df_query) < len(l18_plan):
            print("  Skipping ANOVA due to incomplete data.")
            continue

        for response in response_vars:
            print(f"\n  ANOVA for Response: {response} (Higher is better)")
            formula = f"{response} ~ " + " + ".join([f"C({f})" for f in factor_names])

            try:
                model = ols(formula, data=df_query).fit()
                anova_table = anova_lm(model, typ=2)

                print(anova_table)

                print(f"\n  Post-hoc Tukey HSD for {response} (alpha=0.05)")
                for factor in factor_names:
                    if anova_table.loc[f"C({factor})", "PR(>F)"] < 0.05:
                        print(f"    ... Significant factor: {factor}")
                        tukey_res = pairwise_tukeyhsd(
                            endog=df_query[response],
                            groups=df_query[factor],
                            alpha=0.05
                        )
                        print(tukey_res)

            except ValueError as e:
                print(f"    Could not fit ANOVA model for {response}: {e}")
                print(f"    Data snippet:\n{df_query[[response] + factor_names].head()}")

def main():
    logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    if RESULTS_DIR.exists():
        shutil.rmtree(RESULTS_DIR)
    DATA_DIR.mkdir(parents=True)
    RESULTS_DIR.mkdir(parents=True)

    l18_run_plan = build_l18_run_plan()

    for config in l18_run_plan:
        run_id = config['run_id']
        print(f"\n--- Run {run_id}/{len(l18_run_plan)} ---")
        print(json.dumps(config, indent=2))

        try:
            parquet_path, query_params = generate_dataset(run_id, config)
            df_run_metrics = run_benchmark(run_id, config, parquet_path, query_params)
            save_results(run_id, config, df_run_metrics)
            parquet_path.unlink()

        except Exception as e:
            print(f"Error in run {run_id}: {e}")
            logging.exception(f"Run {run_id} failed.")

    analyze_results()

main()